# A股市场广度 / 阿姆氏指标(TRIN) / 中金所会员持仓 数据拉取

输出 `a_breadth.csv` 和 `cffex_holding.csv`，供 `charts.ipynb` 出图。中途断了重新运行「拉取」那一格即可，会跳过已拉过的日期。

在 GitHub 上运行时 token 从仓库 Secret `TUSHARE_TOKEN` 读取。

**TRIN（Arms Index，标准公式）= (上涨家数 / 下跌家数) / (上涨成交量 / 下跌成交量)**，平盘股不计入。`trin` 为标准口径（成交量）；`trin_amt` 为按成交额的非标准变体，仅作对照。`*_ma10` 为 10 日均线（Arms 原始阈值：<0.8 超买，>1.2 超卖），`*_sum10` 为 10 日累计口径。前缀 `big_` 为当日市值前 300，`small_` 为其余。

接口规则（已按 tushare 文档核对）：
- `daily`：vol 单位手、amount 单位千元，单次上限 6000 行 → offset 翻页
- `daily_basic`：total_mv 单位万元，单次上限 6000 行；`limit_status` 需在 fields 显式请求（2/3 涨停，5/6 跌停）
- `stk_limit`：含基金，单次上限 5800 行 → 仅作兜底且翻页
- `fut_holding`：需 2000 积分，单次上限 2000 行 → 按品种分别查；有品种汇总行时只用汇总行

In [ ]:
# 首次运行先装依赖：
# !pip install tushare pandas


## 拉数据

全部步骤都在下面这一格里，运行它即可。中途断了重新运行，会跳过已拉过的日期。

In [ ]:
# ===================== 拉数据（一整格，可重复运行，自动跳过已拉过的日期） =====================
# 1. 配置

import os, re, sys, time
import pandas as pd
import tushare as ts

TOKEN = os.environ.get("TUSHARE_TOKEN", "")   # GitHub 上从 Secrets 读取；本地运行可在这里填，但别提交
START = "20240101"   # 提前到 2024 年，用于回测博主的 TRIN 阈值规则
END = time.strftime("%Y%m%d")
BIG_N = 300          # 当日总市值前 300 视为"权重/大盘"，其余为"中小盘"
os.makedirs("data", exist_ok=True)
OUT_BREADTH = "data/a_breadth.csv"
OUT_FUT = "data/cffex_holding.csv"
KEY_DATES = ["20260323", "20260721", "20260921", "20260922", "20261022", "20261023"]   # 博主提到的几天

pro = ts.pro_api(TOKEN)
pro._DataApi__token = TOKEN
pro._DataApi__http_url = os.environ.get("TUSHARE_HTTP_URL", "http://jiaoch.site")
assert TOKEN, "没有 token：请在 GitHub 仓库 Settings → Secrets → Actions 里添加 TUSHARE_TOKEN"

# 2. 工具函数

def call(fn, **kw):
    for i in range(5):
        try:
            df = getattr(pro, fn)(**kw)
            time.sleep(0.12)
            return df if df is not None else pd.DataFrame()
        except Exception as e:
            print(f"  {fn} {kw.get('trade_date', '')} 失败：{e}，{2*(i+1)}秒后重试")
            time.sleep(2 * (i + 1))
    return pd.DataFrame()


def call_all(fn, page=5000, **kw):
    """按 offset 翻页，防止单次限量（daily/daily_basic 6000 行）截断数据。"""
    out, off = [], 0
    while True:
        df = call(fn, limit=page, offset=off, **kw)
        if df.empty:
            break
        out.append(df)
        if len(df) < page:
            break
        off += page
    return pd.concat(out, ignore_index=True).drop_duplicates() if out else pd.DataFrame()


def trin(n_up, n_dn, a_up, a_dn):
    if n_dn == 0 or a_up == 0 or a_dn == 0:
        return None
    return (n_up / n_dn) / (a_up / a_dn)


def group_stats(df, prefix):
    up, dn = df.pct_chg > 0, df.pct_chg < 0
    n_up, n_dn = int(up.sum()), int(dn.sum())
    a_up, a_dn = df.loc[up, "amount"].sum() / 1e5, df.loc[dn, "amount"].sum() / 1e5  # 千元 -> 亿元
    tot = df["amount"].sum() / 1e5
    v_up, v_dn = df.loc[up, "vol"].sum() / 1e6, df.loc[dn, "vol"].sum() / 1e6          # 手 -> 百万手
    return {
        f"{prefix}vol": round(df["vol"].sum() / 1e6, 3),
        f"{prefix}up_vol": round(v_up, 3), f"{prefix}dn_vol": round(v_dn, 3),
        # 标准 Arms Index（Richard Arms, 1967）：按成交量（股数/手，单位不影响比值），平盘股不计入
        f"{prefix}trin": trin(n_up, n_dn, v_up, v_dn),
        f"{prefix}n": len(df), f"{prefix}n_up": n_up, f"{prefix}n_dn": n_dn,
        f"{prefix}amt": round(tot, 2), f"{prefix}up_amt": round(a_up, 2), f"{prefix}dn_amt": round(a_dn, 2),
        f"{prefix}up_amt_share": round(a_up / tot, 4) if tot else None,
        # 非标准变体：按成交额（博主口述的口径），偏向大盘高价股，仅作对照
        f"{prefix}trin_amt": trin(n_up, n_dn, a_up, a_dn),
    }


def load(path):
    return pd.read_csv(path, dtype={"trade_date": str}) if os.path.exists(path) else pd.DataFrame()

# 3. 单日市场广度 / TRIN / 大小盘拆分

def day_breadth(d):
    px = call_all("daily", trade_date=d)   # amount 单位：千元
    if px.empty:
        return None
    px = px[px.ts_code.str.endswith((".SH", ".SZ"))].dropna(subset=["pct_chg"])
    # total_mv 单位：万元；limit_status：2涨停 3一字涨停 5跌停 6一字跌停（非默认字段，需显式请求）
    mv = call_all("daily_basic", trade_date=d, fields="ts_code,total_mv,limit_status")
    df = px.merge(mv, on="ts_code", how="left")
    row = {"trade_date": d}
    row.update(group_stats(df, ""))
    row["n_flat"] = row["n"] - row["n_up"] - row["n_dn"]
    row["breadth_up"] = round(row["n_up"] / row["n"], 4)
    ls = pd.to_numeric(df.get("limit_status"), errors="coerce") if "limit_status" in df else None
    if ls is not None and ls.notna().any():
        row["limit_up"] = int(ls.isin([2, 3]).sum())
        row["limit_dn"] = int(ls.isin([5, 6]).sum())
    else:  # 兜底：stk_limit 含基金、单次上限 5800 行，必须翻页
        lim = call_all("stk_limit", trade_date=d)
        if not lim.empty and "up_limit" in lim:
            df = df.merge(lim[["ts_code", "up_limit", "down_limit"]], on="ts_code", how="left")
            row["limit_up"] = int((df.close >= df.up_limit - 0.005).sum())
            row["limit_dn"] = int((df.close <= df.down_limit + 0.005).sum())
    big_codes = set(df.dropna(subset=["total_mv"]).nlargest(BIG_N, "total_mv").ts_code)
    row.update(group_stats(df[df.ts_code.isin(big_codes)], "big_"))
    row.update(group_stats(df[~df.ts_code.isin(big_codes)], "small_"))
    return row

# 4. 单日中金所会员持仓

FUT_PRODUCTS = ["IF", "IH", "IC", "IM"]
FUT_MODE = None   # 由 probe_fut() 决定用哪种调用方式


def fut_query(d, mode, prod):
    if mode == "by_product_ex":
        return pro.fut_holding(trade_date=d, symbol=prod, exchange="CFFEX")
    if mode == "by_product":
        return pro.fut_holding(trade_date=d, symbol=prod)
    return pro.fut_holding(trade_date=d, exchange="CFFEX")


def probe_fut(dates):
    """依次试几种调用方式，打印每种返回了什么，选第一个能拿到 IF 数据的。"""
    print("探测期货持仓接口：")
    for d in list(reversed(dates[-6:-1])) or dates:
        for mode in ["by_product_ex", "by_product", "by_exchange"]:
            try:
                df = fut_query(d, mode, "IF")
                time.sleep(0.12)
            except Exception as e:
                print(f"  {d} {mode}: 出错 {str(e)[:120]}")
                continue
            n = 0 if df is None else len(df)
            syms = sorted(set(df.symbol.astype(str)))[:8] if n and "symbol" in df else []
            print(f"  {d} {mode}: {n} 行，symbol 样例 {syms}")
            if n and "symbol" in df and df.symbol.astype(str).str.startswith("IF").any():
                print(f"  → 使用 {mode}")
                return mode
    return None


def day_fut(d):
    # symbol 可能是品种代码(IF)也可能是合约(IF2611)：有品种汇总行就只用汇总行，否则把各合约加总，避免重复相加。
    if FUT_MODE == "by_exchange":
        raw = call("fut_holding", trade_date=d, exchange="CFFEX")
        parts = {p: raw for p in FUT_PRODUCTS}
    else:
        extra = {"exchange": "CFFEX"} if FUT_MODE == "by_product_ex" else {}
        parts = {p: call("fut_holding", trade_date=d, symbol=p, **extra) for p in FUT_PRODUCTS}
    rows = []
    for prod, fh in parts.items():
        if fh.empty or "symbol" not in fh:
            continue
        sym = fh.symbol.astype(str)
        agg = fh[sym == prod]
        src = "product" if not agg.empty else "contracts_sum"
        fh = agg.copy() if not agg.empty else fh[sym.str.match(rf"^{prod}\d{{4}}$")].copy()
        if fh.empty:
            continue
        for c in ["long_hld", "short_hld", "vol"]:
            fh[c] = pd.to_numeric(fh[c], errors="coerce").fillna(0)
        g = fh.groupby("broker", as_index=False)[["long_hld", "short_hld", "vol"]].sum()
        g["product"], g["trade_date"], g["source"] = prod, d, src
        rows.extend(g.to_dict("records"))
    return rows


def add_smoothing(b, n=10):
    """Arms 原始用法看 10 日均线：<0.8 超买，>1.2 超卖。
    另算一个 10 日累计口径：分子分母各自先求 10 日和再相除，避免单日极端值主导均值。"""
    for pre in ["", "big_", "small_"]:
        for col in ["trin", "trin_amt"]:
            if pre + col in b:
                b[f"{pre}{col}_ma{n}"] = b[pre + col].rolling(n).mean()
        r = {k: b[pre + k].rolling(n).sum() for k in ["n_up", "n_dn", "up_vol", "dn_vol"] if pre + k in b}
        if len(r) == 4:
            b[f"{pre}trin_sum{n}"] = (r["n_up"] / r["n_dn"]) / (r["up_vol"] / r["dn_vol"])
    return b

# 5. 拉取（可断点续跑，2024 年至今约需 30–40 分钟）

cal = call("trade_cal", exchange="SSE", start_date=START, end_date=END, is_open="1")
dates = sorted(cal["cal_date"].astype(str))
print(f"交易日 {len(dates)} 天：{dates[0]} ~ {dates[-1]}")

# 旧版程序生成的数据缺少成交量字段，不能混用：改名备份后重新拉取
if os.path.exists(OUT_BREADTH):
    have = set(pd.read_csv(OUT_BREADTH, nrows=1).columns)
    if not {"up_vol", "dn_vol", "trin_amt"} <= have:
        os.replace(OUT_BREADTH, OUT_BREADTH.replace(".csv", "_old.csv"))
        print("发现旧版数据（缺少新字段），已改名为 *_old.csv，重新拉取")

FUT_MODE = probe_fut(dates)
if FUT_MODE is None:
    print("⚠ 期货持仓接口几种调用方式都没拿到数据（上面是每种方式的返回），本次跳过期货部分，其余照常。")

b_old, f_old = load(OUT_BREADTH), load(OUT_FUT)
b_done = set(b_old.trade_date) if not b_old.empty else set()
f_done = set(f_old.trade_date) if not f_old.empty else set()
b_rows, f_rows = [], []

for i, d in enumerate(dates, 1):
    if d not in b_done:
        r = day_breadth(d)
        if r:
            b_rows.append(r)
    if FUT_MODE and d not in f_done:
        f_rows.extend(day_fut(d))
    if i % 20 == 0 or i == len(dates):
        print(f"  进度 {i}/{len(dates)}  {d}")
        b_all = pd.concat([b_old, pd.DataFrame(b_rows)], ignore_index=True)
        f_all = pd.concat([f_old, pd.DataFrame(f_rows)], ignore_index=True)
        if not b_all.empty:
            b_all.drop_duplicates("trade_date", keep="last").sort_values("trade_date").to_csv(OUT_BREADTH, index=False, encoding="utf-8-sig")
        if not f_all.empty:
            f_all.drop_duplicates(["trade_date", "product", "broker"], keep="last").sort_values("trade_date").to_csv(OUT_FUT, index=False, encoding="utf-8-sig")

# 6. 并入指数收盘价

b = load(OUT_BREADTH)
for code, name in [("000001.SH", "sh"), ("000300.SH", "hs300"), ("000852.SH", "zz1000")]:
    idx = call("index_daily", ts_code=code, start_date=START, end_date=END)
    if not idx.empty:
        idx = idx[["trade_date", "close", "pct_chg"]].rename(columns={"close": f"{name}_close", "pct_chg": f"{name}_pct"})
        b = b.drop(columns=[c for c in idx.columns if c != "trade_date" and c in b], errors="ignore")
        b = b.merge(idx.astype({"trade_date": str}), on="trade_date", how="left")
b = add_smoothing(b.sort_values("trade_date"))
b.to_csv(OUT_BREADTH, index=False, encoding="utf-8-sig")

# 7. 博主提到的几天

cols = ["trade_date", "breadth_up", "limit_up", "trin", "trin_amt", "trin_ma10", "up_amt_share", "amt", "big_trin", "big_up_amt_share", "small_trin", "sh_close"]
print("\n博主提到的几天：")
print(b[b.trade_date.isin(KEY_DATES)][[c for c in cols if c in b]].to_string(index=False))
print(f"\n完成，数据已写入 {OUT_BREADTH} 和 {OUT_FUT}。")
